In [1]:
import numpy as np
import pandas as pd 

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline
)

In [2]:
DATA_DIR = 'filtered_arabic_sentiment_reviews.csv'

In [3]:
df = pd.read_csv(DATA_DIR, nrows= 100000)

In [4]:
ds = Dataset.from_dict({
    "text": df['content'].values,
    "label": df['label'].values
})

In [5]:
print(f"Review : {ds['text'][2]}\nRating : {ds['label'][2]}")

Review : واحد أقل من العميل مدى الحياة: أدى رأس سيئ التصميم في هذه السلسلة إلى انهيار الخلاط حرفيًا في الطعام الذي كنت أقوم به.أقسمت بالمساعدات المطبخ عندما عملت في المطاعم.أقنعت زوجتي بالتواصل معي في شراء المحترف.الآن ولائي يجعلني أبدو مثل أحمق.أطلقنا على KitchenAid ، لكنهم فشلوا في الوقوف إلى جانب الخلاط - وفقدوا مدافعًا قويًا عن علامتهم التجارية.
Rating : 0


# set up tokenizer


In [6]:
BASE_MODEL = "distilbert-base-multilingual-cased"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
def preprocess_function(examples) : 
    return tokenizer(examples['text'] , truncation = True)

In [8]:
ds = ds.map(preprocess_function , batched = True)
ds = ds.remove_columns("label")

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [9]:
train_testvalid = ds.train_test_split(test_size=0.2, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.3, seed=42)

In [10]:
train_ds = train_testvalid['train']
val_ds = test_valid['train']
test_ds = test_valid['test']
print(f"train dataset : {len(train_ds)}")
print(f"validation dataset : {len(val_ds)}")
print(f"test dataset : {len(test_ds)}")

train dataset : 80000
validation dataset : 14000
test dataset : 6000


# MASKED LANGUAGE MODELING (MLM) - Domain Adaptation


In [11]:
# (DataCollatorForLanguageModeling) token masking, we randomly mask 15% of the tokens in a sentence.
# It might happen that part of a word will be masked
# to apply whole-word masking (DataCollatorForWholeWordMask)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [12]:
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [13]:
# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

Total parameters     : 135,445,755
Trainable parameters : 135,445,755


In [14]:
args = TrainingArguments(
    "mlm_model",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_strategy = "epoch",
    per_device_train_batch_size = 8, 
    per_device_eval_batch_size = 8,
    num_train_epochs = 10, 
    learning_rate = 2e-5, 
    weight_decay = 0.01,
    report_to="none"
)

In [15]:
trainer = Trainer(
    model = model,
    processing_class  = tokenizer,
    data_collator=data_collator,
    args = args,
    train_dataset = train_ds,
    eval_dataset = val_ds
)

In [16]:
# Save pre-trained tokenizer
tokenizer.save_pretrained("mlm")

('mlm\\tokenizer_config.json', 'mlm\\tokenizer.json')

In [17]:
results = trainer.train()

Epoch,Training Loss,Validation Loss
1,1.941773,1.629713
2,1.652904,1.499957
3,1.541367,1.422287
4,1.474520,1.378927
5,1.423051,1.346820
6,1.385845,1.316057
7,1.353139,1.295489
8,1.329875,1.275393
9,1.312387,1.264660
10,1.301833,1.256893


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
# Save updated model
model.save_pretrained("mlm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
# test the pre-trained model
mask_filler = pipeline("fill-mask" , model = BASE_MODEL)
preds = mask_filler("هذا المنتج [MASK] جداً!")

# Print results
for pred in preds:
    print(f">>> {pred["sequence"]}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

>>> هذا المنتج معروف جداً!
>>> هذا المنتج ليس جداً!
>>> هذا المنتج يعمل جداً!
>>> هذا المنتج غير جداً!
>>> هذا المنتج هو جداً!


In [20]:
# test the model after Continued Pretraining with MLM
mask_filler = pipeline("fill-mask" , model = 'mlm')
preds = mask_filler("هذا المنتج [MASK] جداً!")

# Print results
for pred in preds:
    print(f">>> {pred["sequence"]}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

>>> هذا المنتج سعيد جداً!
>>> هذا المنتج كبير جداً!
>>> هذا المنتج مناسب جداً!
>>> هذا المنتج ليس جداً!
>>> هذا المنتج يعمل جداً!
